# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [ ]:
# @title Setup (chạy ô này trước)
# Clone fork chứa labkit.evidence; giữ nguyên checkout trong suốt thí nghiệm frozen.
import os, pathlib, subprocess, sys

REPO = "https://github.com/Nguyen-Sam-sheep-zzz/Day21-Track3-Finetuning-Lab.git"
BRANCH = "feature/lab21-finetuning"
HERE = pathlib.Path.cwd()
REPO_DIR = HERE if (HERE / ".git").exists() else HERE / "Day21-Track3-Finetuning-Lab"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--single-branch",
                    REPO, str(REPO_DIR)], check=True)
if not (REPO_DIR / ".git").exists():
    raise RuntimeError("Setup cần checkout Git riêng; dùng runtime/thư mục mới trước NB2.")
origin = subprocess.run(["git", "config", "--get", "remote.origin.url"], cwd=REPO_DIR,
                        capture_output=True, text=True, check=True).stdout.strip()
branch = subprocess.run(["git", "branch", "--show-current"], cwd=REPO_DIR,
                        capture_output=True, text=True, check=True).stdout.strip()
if origin.removesuffix(".git") != REPO.removesuffix(".git") or branch != BRANCH:
    raise RuntimeError("Checkout không đúng fork/branch. Chọn checkout đúng trước NB2; không đổi code giữa run frozen.")
if not (REPO_DIR / "src" / "labkit" / "evidence.py").exists():
    raise RuntimeError("Checkout thiếu labkit.evidence. Dùng bản feature/lab21-finetuning đã push trước NB2.")
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
# Setup không pull/fetch/checkout/reset trên checkout đã tồn tại.

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

os.environ.setdefault("COMPUTE_TIER", "T4")
import torch
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True, check=True).stdout.strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core pipeline — NB1 → NB5
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = "8"         # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null
